# Lab 2 - SPARK RDD
## Bài tập về nhà: OpenFlights

#### MSSV: 23521720
#### Họ tên: Nguyễn Minh Tuấn



## A. Chuẩn bị dữ liệu


In [ ]:
from google.colab import files
import os
import shutil

# Thu muc luu du lieu
data_dir = "/content/data/openflights"
os.makedirs(data_dir, exist_ok=True)

# Upload 5 file .dat
uploaded = files.upload()

for filename in uploaded:
    shutil.move(
        f"/content/{filename}",
        f"{data_dir}/{filename}"
    )

# Kiem tra du 5 file
required_files = [
    "routes.dat",
    "airports.dat",
    "airlines.dat",
    "planes.dat",
    "countries.dat"
]

for filename in required_files:
    path = os.path.join(data_dir, filename)
    if os.path.exists(path):
        print(f"OK: {filename}")
    else:
        print(f"THIEU: {filename}")

Saving airlines.dat to airlines.dat
Saving airports.dat to airports.dat
Saving countries.dat to countries.dat
Saving planes.dat to planes.dat
Saving routes.dat to routes.dat
OK: routes.dat
OK: airports.dat
OK: airlines.dat
OK: planes.dat
OK: countries.dat


In [ ]:
import csv
from pathlib import Path
from operator import add

try:
    from pyspark.sql import SparkSession
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.1'])
    from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Lab2_OpenFlights').master('local[*]').getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

thu_muc = [Path('data/openflights'), Path('../data/openflights'),
            Path('/content/data/openflights')]
OF = next((p.resolve() for p in thu_muc if (p / 'routes.dat').exists()), None)
if OF is None:
    raise FileNotFoundError('Khong tim thay thu muc data/openflights')
print('Duong dan:', OF)

Duong dan: /content/data/openflights


## B. Bài tập bắt buộc

### Bài 1. Đọc dữ liệu và kiểm tra

In [ ]:
def tach_csv(line):
    return next(csv.reader([line]))

routes = sc.textFile(str(OF / 'routes.dat')).map(lambda x: x.split(',')).cache()
airports = sc.textFile(str(OF / 'airports.dat')).map(tach_csv).cache()
airlines = sc.textFile(str(OF / 'airlines.dat')).map(tach_csv).cache()

for ten, rdd in [('routes', routes), ('airports', airports), ('airlines', airlines)]:
    print(ten, ':', rdd.count(), 'dong,', len(rdd.first()), 'cot')

iata_hop_le = lambda p: len(p[4]) == 3 and p[4].isalpha()
print('San bay co ma IATA hop le:', airports.filter(iata_hop_le).count())
print('Ten san bay co chua dau phay:', airports.filter(lambda p: ',' in p[1]).count())

bang_sb = airports.filter(iata_hop_le) \
    .map(lambda p: (p[4], (p[1], p[2], p[3]))).collectAsMap()
bang_hang = airlines.map(lambda p: (p[0], (p[1], p[3]))).collectAsMap()
bc_sb = sc.broadcast(bang_sb)
bc_hang = sc.broadcast(bang_hang)

routes : 67663 dong, 9 cot
airports : 7698 dong, 14 cot
airlines : 6162 dong, 8 cot
San bay co ma IATA hop le: 6071
Ten san bay co chua dau phay: 16


### Bài 2. Sân bay bận rộn nhất

In [ ]:
luot_di = routes.map(lambda p: (p[2], 1)).reduceByKey(add).cache()
luot_den = routes.map(lambda p: (p[4], 1)).reduceByKey(add)
bang_luot_den = luot_den.collectAsMap()

print('So san bay co duong bay di:', luot_di.count())
print('So san bay co duong bay den:', luot_den.count())
print('Tong duong bay di:', luot_di.values().sum())
print('Tong duong bay den:', luot_den.values().sum())
print('\nTop 10 san bay co nhieu duong bay di:')

for ma, so in luot_di.takeOrdered(10, key=lambda x: (-x[1], x[0])):
    sb = bc_sb.value.get(ma, ('?', '?', '?'))
    print(f'{ma:4} {so:5}    {sb[1]}, {sb[2]}')

So san bay co duong bay di: 3409
So san bay co duong bay den: 3418
Tong duong bay di: 67663
Tong duong bay den: 67663

Top 10 san bay co nhieu duong bay di:
ATL    915    Atlanta, United States
ORD    558    Chicago, United States
PEK    535    Beijing, China
LHR    527    London, United Kingdom
CDG    524    Paris, France
FRA    497    Frankfurt, Germany
LAX    492    Los Angeles, United States
DFW    469    Dallas-Fort Worth, United States
JFK    456    New York, United States
AMS    453    Amsterdam, Netherlands


### Bài 3. Số hãng bay khai thác mỗi sân bay

In [ ]:
cap_san_bay_hang = routes.map(lambda p: (p[2], p[0])).distinct().cache()
so_hang = cap_san_bay_hang.map(lambda x: (x[0], 1)).reduceByKey(add)
bang_luot_di = luot_di.collectAsMap()

print('So cap (san bay, hang bay) khac nhau:', cap_san_bay_hang.count())
print('So san bay chi co 1 hang:', so_hang.filter(lambda x: x[1] == 1).count())
print('So hang khai thac tai ATL:', so_hang.filter(lambda x: x[0] == 'ATL').first()[1])
print('\nTop 10 san bay:')

for ma, so in so_hang.takeOrdered(10, key=lambda x: (-x[1], x[0])):
    sb = bc_sb.value.get(ma, ('?', '?', '?'))
    print(f'{ma:4} {so:3} hang  {bang_luot_di[ma]:4} duong bay   {sb[1]}, {sb[2]}')

So cap (san bay, hang bay) khac nhau: 19288
So san bay chi co 1 hang: 1313
So hang khai thac tai ATL: 37

Top 10 san bay:
CDG  108 hang   524 duong bay   Paris, France
FRA  100 hang   497 duong bay   Frankfurt, Germany
BKK   98 hang   326 duong bay   Bangkok, Thailand
FCO   92 hang   331 duong bay   Rome, Italy
LHR   86 hang   527 duong bay   London, United Kingdom
DXB   84 hang   356 duong bay   Dubai, United Arab Emirates
HKG   83 hang   355 duong bay   Hong Kong, Hong Kong
SIN   83 hang   408 duong bay   Singapore, Singapore
PEK   81 hang   535 duong bay   Beijing, China
AMS   79 hang   453 duong bay   Amsterdam, Netherlands


### Bài 4. Thống kê theo loại máy bay

In [ ]:
co_may_bay = routes.filter(lambda p: p[8].strip() != '')
thieu_may_bay = routes.filter(lambda p: p[8].strip() == '')
cap_duong_bay_may_bay = routes.flatMap(lambda p: [(ma, 1) for ma in p[8].split()]).cache()
so_duong_bay_theo_may = cap_duong_bay_may_bay.reduceByKey(add)

print('Chuyen co khai bao may bay :', co_may_bay.count())
print('Chuyen khong khai bao may bay :', thieu_may_bay.count())
print('Tong cap (duong bay, may bay):', cap_duong_bay_may_bay.count())
print('So loai may bay khac nhau:', so_duong_bay_theo_may.count())
print('Nhieu loai may bay nhat tren 1 chuyen:',
      routes.map(lambda p: len(p[8].split())).max())
print('\nTop 12 loai may bay:')

for ma, so in so_duong_bay_theo_may.takeOrdered(12, key=lambda x: (-x[1], x[0])):
    print(f'{ma:5} {so}')

Chuyen co khai bao may bay : 67645
Chuyen khong khai bao may bay : 18
Tong cap (duong bay, may bay): 93231
So loai may bay khac nhau: 168
Nhieu loai may bay nhat tren 1 chuyen: 9

Top 12 loai may bay:
320   15514
738   10406
319   7885
321   3677
73H   3351
737   2787
73W   2502
CRJ   2224
E90   1970
AT7   1635
73G   1592
333   1589


### Bài 5. Số đường bay theo quốc gia

In [ ]:
tra_duoc_di = routes.filter(lambda p: p[2] in bc_sb.value).cache()
khong_tra_duoc_di = routes.filter(lambda p: p[2] not in bc_sb.value).cache()

so_duong_bay_qg = tra_duoc_di.map(
    lambda p: (bc_sb.value[p[2]][2], 1)
).reduceByKey(add)

print('So quoc gia:', so_duong_bay_qg.count())
print('Chuyen khong tra duoc san bay di:', khong_tra_duoc_di.count())
print('So ma san bay khong tra duoc san bay di:',
      khong_tra_duoc_di.map(lambda p: p[2]).distinct().count())
print('Kiem tra:', so_duong_bay_qg.values().sum(), '+',
      khong_tra_duoc_di.count(), '=', routes.count())
print('\nTop 12 quoc gia:')

for qg, so in so_duong_bay_qg.takeOrdered(12, key=lambda x: (-x[1], x[0])):
    print(f'{qg:23} {so}')

So quoc gia: 225
Chuyen khong tra duoc san bay di: 406
So ma san bay khong tra duoc san bay di: 157
Kiem tra: 67257 + 406 = 67663

Top 12 quoc gia:
United States           13100
China                   8160
United Kingdom          2663
Spain                   2531
Germany                 2352
France                  1930
Canada                  1852
Russia                  1843
Italy                   1776
India                   1507
Brazil                  1398
Japan                   1286


### Bài 6. Mạng bay Việt Nam

In [ ]:
def la_san_bay_vn(ma):
    sb = bc_sb.value.get(ma)
    return sb is not None and sb[2] == 'Vietnam'

di_vn = routes.filter(lambda p: la_san_bay_vn(p[2])).cache()
den_vn = routes.filter(lambda p: la_san_bay_vn(p[4]))
noi_dia_vn = di_vn.filter(lambda p: la_san_bay_vn(p[4]))
quoc_te_vn = di_vn.filter(lambda p: not la_san_bay_vn(p[4]))
chua_ro_vn = di_vn.filter(lambda p: p[4] not in bc_sb.value)

print('Chuyen bay di tu VN:', di_vn.count())
print('Chuyen bay den VN:', den_vn.count())
print('Chuyen bay noi dia:', noi_dia_vn.count())
print('Chuyen bay quoc te di:', quoc_te_vn.count())
print('Trong do khong tra duoc san bay den:', chua_ro_vn.count())

san_bay_vn = di_vn.map(lambda p: (p[2], 1)).reduceByKey(add).collect()
print('\nSo san bay co chuyen bay di:', len(san_bay_vn), '\n')
for ma, so in sorted(san_bay_vn, key=lambda x: (-x[1], x[0])):
    print(f'{ma:4} {so:3}  {bc_sb.value[ma][1]}')

Chuyen bay di tu VN: 288
Chuyen bay den VN: 288
Chuyen bay noi dia: 126
Chuyen bay quoc te di: 162
Trong do khong tra duoc san bay den: 1

So san bay co chuyen bay di: 20 

SGN  109  Ho Chi Minh City
HAN   87  Hanoi
DAD   33  Danang
BMV    7  Buonmethuot
CXR    7  Nha Trang
VII    7  Vinh
DLI    5  Dalat
PQC    5  Phuquoc
HPH    4  Haiphong
HUI    4  Hue
PXU    3  Pleiku
UIH    3  Phucat
VCA    3  Can Tho
TBB    2  Tuy Hoa
VCL    2  Chu Lai
VCS    2  Conson
VDH    2  Dong Hoi
CAH    1  Ca Mau
DIN    1  Dienbienphu
VKG    1  Rach Gia


In [ ]:
hang_vn = di_vn.map(lambda p: (p[1], 1)).reduceByKey(add).collect()
print('So hang bay:', len(hang_vn), '\n')
for ma_hang, so in sorted(hang_vn, key=lambda x: (-x[1], x[0])):
    ten, iata = bc_hang.value.get(ma_hang, ('Khong ro', '?'))
    print(f'{iata:4} {so:3}  {ten}')

So hang bay: 45 

VN   139  Vietnam Airlines
VJ    32  Royal Air Cambodge
BL    18  Jetstar Pacific
JL     8  Japan Airlines
KE     5  Korean Air
K6     4  Cambodia Angkor Air (K6)
MU     4  China Eastern Airlines
CZ     4  China Southern Airlines
SQ     4  Singapore Airlines
AZ     4  Alitalia
AF     3  Air France
CX     3  Cathay Pacific
CI     3  China Airlines
OZ     3  Asiana Airlines
QV     3  Lao Airlines
QR     3  Qatar Airways
MI     3  SilkAir
SU     2  Aeroflot Russian Airlines
5J     2  Cebu Pacific
KA     2  Dragonair
BR     2  EVA Air
AA     2  American Airlines
HX     2  Hong Kong Airlines
NX     2  Air Macau
NH     2  All Nippon Airways
MH     2  Malaysia Airlines
AE     2  Mandarin Airlines
QF     2  Qantas
TR     2  Tiger Airways
TG     2  Thai Airways International
FD     2  Thai AirAsia
VA     2  Virgin Australia
AK     2  AirAsia
B7     2  Uni Air
KC     1  Air Astana
EK     1  Emirates
EY     1  Etihad Airways
3K     1  Jetstar Asia Airways
JT     1  Lion Mentari 

## C. Bài tập tự chọn

### Bài 7. Đường bay nội địa và quốc tế trên thế giới

In [ ]:
def phan_loai(p):
    san_bay_di = bc_sb.value.get(p[2])
    san_bay_den = bc_sb.value.get(p[4])
    if san_bay_di is None or san_bay_den is None:
        return 'Khong tra duoc'
    if san_bay_di[2] == san_bay_den[2]:
        return 'Noi dia'
    return 'Quoc te'

thong_ke = routes.map(lambda p: (phan_loai(p), 1)) \
    .reduceByKey(add).collectAsMap()

noi_dia = thong_ke.get('Noi dia', 0)
quoc_te = thong_ke.get('Quoc te', 0)
khong_tra_duoc = thong_ke.get('Khong tra duoc', 0)

print('Noi dia:', noi_dia)
print('Quoc te:', quoc_te)
print('Khong tra duoc:', khong_tra_duoc)
print('Tong:', noi_dia + quoc_te + khong_tra_duoc)
print(f'Ti le quoc te: {quoc_te / (noi_dia + quoc_te) * 100:.1f}%')

Noi dia: 32167
Quoc te: 34767
Khong tra duoc: 729
Tong: 67663
Ti le quoc te: 51.9%
